# 1. Importación de Librerías y Funciones
#
Importamos las librerías necesarias para el modelado y la evaluación, incluyendo:
- `scanpy` y `pandas` para la manipulación de datos.
 - `sklearn` para el modelado (RandomForest, train_test_split, métricas).
- `joblib` para guardar nuestros modelos entrenados.
 - Nuestra función de ploteo personalizada desde la carpeta `src`.

In [ ]:
import scanpy as sc
import os
import sys

# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.append('../src')
from models import train_and_evaluate_model
from plotting import plot_confusion_matrix

# 2. Carga del Dataset Procesado

Cargamos el dataset limpio y anotado que generamos en el notebook anterior. También definimos las rutas de salida para guardar las figuras y los modelos que generemos.


Definimos las rutas relativas

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
PROCESSED_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
OUTPUTS_PATH = '../outputs/'
FIGURES_PATH = os.path.join(OUTPUTS_PATH, 'figures')
MODELS_PATH = os.path.join(OUTPUTS_PATH, 'models')

Creamos las carpetas de salida si no existen

In [ ]:
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

Cargamos los datos

In [ ]:
adata_final = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, PROCESSED_FILENAME))

print("Dataset procesado cargado exitosamente:")
print(adata_final)

# Experimento 1: Modelo Base con Todos los Genes

 Nuestro primer objetivo es establecer un rendimiento de referencia (baseline). Entrenaremos un clasificador Random Forest utilizando todos los genes disponibles como características para ver cómo de bien puede distinguir los tipos celulares sin ninguna optimización.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 1: Modelo Base con Todos los Genes ---")

1. Preparar los datos X (features) e y (target)

In [ ]:
X_all_genes = adata_final.X
y = adata_final.obs['cell_type']

2. Dividir en set de entrenamiento y test

In [ ]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all_genes,
    y,
    test_size=0.2,
    random_state=42,  # Usar un estado fijo para reproducibilidad
    stratify=y        # Esencial para mantener la proporción de clases
)

3. Entrenar el modelo

In [ ]:
model_rf_all = RandomForestClassifier(random_state=42, n_jobs=-1) # n_jobs=-1 usa todos los cores
print("Entrenando el modelo con todos los genes...")
model_rf_all.fit(X_train_all, y_train)
print("Entrenamiento completado.")

4. Evaluar el modelo

In [ ]:
print("\n--- Evaluación del Modelo Base ---")
y_pred_all = model_rf_all.predict(X_test_all)
accuracy_all = accuracy_score(y_test, y_pred_all)
report_all = classification_report(y_test, y_pred_all)
cm_all = confusion_matrix(y_test, y_pred_all)

print(f"Precisión (Accuracy): {accuracy_all * 100:.2f}%")
print("\nReporte de Clasificación:")
print(report_all)

5. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(
    cm_all, model_rf_all.classes_,
    title='Matriz de Confusión - RF (Todos los genes)',
    save_path=os.path.join(FIGURES_PATH, 'cm_rf_all_genes.png')
)

joblib.dump(model_rf_all, os.path.join(MODELS_PATH, 'rf_all_genes.joblib'))
print(f"Modelo y figura guardados en la carpeta '{OUTPUTS_PATH}'")


# Experimento 2: Selección de Características Biológicamente Informada
El primer modelo mostró debilidades, especialmente en la distinción de clases biológicamente similares. Nuestra hipótesis es que el rendimiento puede mejorar si forzamos al modelo a centrarse únicamente en los genes más distintivos de cada tipo celular (genes marcadores).

Usaremos `scanpy.tl.rank_genes_groups` para identificar estos genes.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 2: Selección de Genes Marcadores ---")

1. Encontrar genes marcadores

In [ ]:
print("Calculando genes marcadores...")
sc.tl.rank_genes_groups(adata_final, groupby='cell_type', method='t-test')

2. Visualizar los marcadores para confirmar

In [ ]:
print("Visualizando los mejores marcadores...")
sc.pl.rank_genes_groups_dotplot(adata_final, n_genes=4, show=True, save="_marker_genes.png")

3. Extraer la lista de genes marcadores para el modelo

In [ ]:
marker_genes_df = pd.DataFrame(adata_final.uns['rank_genes_groups']['names'])
top_n_genes = 25  # Podemos ajustar este número

marker_genes_list = []
for col in marker_genes_df.columns:
    marker_genes_list.extend(marker_genes_df[col].head(top_n_genes))

marker_genes_list = list(set(marker_genes_list))
print(f"\nSe han seleccionado {len(marker_genes_list)} genes marcadores únicos para el nuevo modelo.")

# Experimento 3: Modelo Optimizado con Genes Marcadores

Ahora, re-entrenaremos el clasificador Random Forest, pero esta vez utilizando únicamente el subconjunto de genes marcadores que acabamos de identificar. Compararemos su rendimiento directamente con el del modelo base.


In [ ]:
print("\n--- INICIANDO EXPERIMENTO 3: Modelo Optimizado con Genes Marcadores ---")

1. Preparar los nuevos datos X (features)

In [ ]:
X_markers = adata_final[:, marker_genes_list].X

2. Dividir los datos (usando la misma y, y mismos parámetros de split)

In [ ]:
X_train_markers, X_test_markers, y_train_markers, y_test_markers = train_test_split(
    X_markers,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

3. Entrenar el nuevo modelo

In [ ]:
model_rf_markers = RandomForestClassifier(random_state=42, n_jobs=-1)
print("Entrenando el modelo con genes marcadores...")
model_rf_markers.fit(X_train_markers, y_train_markers)
print("Entrenamiento completado.")

4. Evaluar el modelo optimizado

In [ ]:
print("\n--- Evaluación del Modelo Optimizado ---")
y_pred_markers = model_rf_markers.predict(X_test_markers)
accuracy_markers = accuracy_score(y_test_markers, y_pred_markers)
report_markers = classification_report(y_test_markers, y_pred_markers)
cm_markers = confusion_matrix(y_test_markers, y_pred_markers)

print(f"Precisión (Accuracy): {accuracy_markers * 100:.2f}%")
print("\nReporte de Clasificación:")
print(report_markers)

5. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(
    cm_markers, model_rf_markers.classes_,
    title='Matriz de Confusión - RF (Genes Marcadores)',
    save_path=os.path.join(FIGURES_PATH, 'cm_rf_marker_genes.png')
)

joblib.dump(model_rf_markers, os.path.join(MODELS_PATH, 'rf_marker_genes.joblib'))
print(f"Modelo y figura optimizados guardados en la carpeta '{OUTPUTS_PATH}'")


# Experimento 4: Benchmark con XGBoost
Para asegurar que hemos elegido el mejor enfoque, comparamos nuestro Random Forest con XGBoost, un algoritmo de gradient boosting a menudo más potente.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 4: Modelo XGBoost con Genes Marcadores ---")

1. Entrenar el modelo XGBoost

In [ ]:
model_xgb_obj, report_xgb, cm_xgb = train_and_evaluate_model(
    'xgb',
    X_train_markers, y_train,
    X_test_markers, y_test,
    output_dir='../outputs',
    model_name='xgb_marker_genes'
)
print("\nReporte de Clasificación (XGBoost):")
print(report_xgb)

 Extraemos las clases del label encoder para el plot

In [ ]:
plot_confusion_matrix(cm_xgb, model_xgb_obj['label_encoder'].classes_, title='Matriz de Confusión - XGBoost (Genes Marcadores)',
                    save_path='../outputs/figures/cm_xgb_marker_genes.png')

# Experimento 5: Benchmark con MLP

In [ ]:
model_mlp, report_mlp, cm_mlp = train_and_evaluate_model(
    'mlp',
    X_train_markers, y_train,
    X_test_markers, y_test,
    output_dir='../outputs',
    model_name='mlp_marker_genes',
    # Podemos pasar hiperparámetros específicos si queremos
    hidden_layer_sizes=(128, 64, 32),
    max_iter=500 
)
print("\nReporte de Clasificación (MLP):")
print(report_mlp)

In [ ]:
plot_confusion_matrix(cm_mlp, model_mlp.classes_, title='Matriz de Confusión - MLP (Genes Marcadores)',
                    save_path='../outputs/figures/cm_mlp_marker_genes.png')

# 4. Comparación de Modelos y Conclusiones

En esta sección final, comparamos directamente el rendimiento de ambos modelos para extraer conclusiones.

**Resultados clave:**
 - El modelo base con todos los genes alcanzó una precisión de XX.XX%.
- El modelo optimizado con genes marcadores alcanzó una precisión de YY.YY%.
- **Análisis de la mejora:** El mayor impacto se observó en la clase `epithelial cell`, donde el F1-score aumentó de A a B, y la confusión con `malignant cell` se redujo significativamente.

**Conclusión:** La selección de características basada en la expresión génica diferencial (genes marcadores) es una 
#estrategia altamente efectiva para mejorar no solo la precisión global, sino, más importante aún, la capacidad del modelo 
#para distinguir entre tipos celulares biológicamente similares, lo que aumenta su fiabilidad para futuras aplicaciones de deconvolución.